# Pipeline
0. Get the list of required swc files
1. Load labels parquet
2. Import the swc file
3. simplify swc file
4. attach synapse labels + neuron type
5. save simplified file
6. convert to json for find-clumpiness
7. save json file
8. calculate clumpiness for each internal node
9. attach results to the labeled swc file
10. save results.

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [2]:
import os
import json
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from joblib import Parallel, delayed
from scripts.preprocessing import simplify_swc_topology, swc2json, get_neurons_info
from scripts.processing import generate_internal_subtrees

if False:
    # Type data located in the 
    path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
    swc_labels = pd.read_feather(path_swc_labels)

    required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
    swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

----

Objective -> attach the clumpiness score to the simplified SWC data.
1. [ ] Read the clumpiness file.
2. [ ] Get list of relevent neurons (from the parquet).
3. [ ] Get list of existing SWC files from the simplified_swc folder.
4. [ ] Define joblib for multitasking while itirating.
5. [ ] Merge simplified_swc with clumpiness.
6. [ ] Save input in another folder.

In [ ]:
###########
# > Imports
import pyarrow.dataset as ds
from tqdm import tqdm
import pandas as pd
import numpy as np
import os


################
# > Define paths
path_simple1 = os.path.join("data", "input_swc", "simplified")
path_simple2 = os.path.join("data", "input_swc", "simplified_swc")
path_og = os.path.join("data", "input_swc", "sk_lod1_783_healed")
path_2process = path_simple1


############################
# > Read the clumpiness file.
# Create a dataset object (does not load data into memory yet)
dataset = ds.dataset(os.path.join("data", "unified_clumpiness.parquet"), format="parquet")


####################################################
# > Get list of relevent neurons (from the parquet).
#### EXAMPLE #### table = dataset.to_table(filter=ds.field("age") > 30, columns=["name", "age"] )  
table_neurons = dataset.to_table(columns=["neuron_id"]).to_pandas()
df_neuronsids = table_neurons.neuron_id.unique() 

# List of SWC files
swc_simplified_dir = [i.split(".")[0] for i in os.listdir(path_simple1)]
swc_og_dir = [i.split(".")[0] for i in os.listdir(path_og)]

# Nuerons to process
relv_files = np.intersect1d(df_neuronsids, swc_simplified_dir)


######################################################################
# > Itirating ove rthe SWCs files and attaching it's relevent metadata
save_path = os.path.join("data", "output_results")
for i in tqdm(relv_files):
    output_path = os.path.join(save_path, f"{i}.csv")

    if os.path.exists(output_path) is False:
        # Defining required path and loading swc + labels file
        i_path = os.path.join(path_2process, f"{i}.csv")
        i_swc = pd.read_csv(i_path, index_col=0)
        i_label = dataset.to_table(filter=(ds.field("neuron_id") == i)).to_pandas() # & 
                                          #(ds.field("property1") == "pre") & 
                                          #(ds.field("property2") == "post")).to_pandas()
        i_label.node_id = i_label.node_id.astype("int")

        # creating a unified labels column names 'label'
        i_label.insert(loc = 2, 
                    column = "label",
                    value = i_label.property1 + "_" + i_label.property2)

        # Use pivot_table instead of pivot, and specify the index
        flipped_labels =i_label.pivot_table(index=['neuron_id', 'node_id'], 
                                            columns='label',
                                            values='value',
                                            aggfunc='first').reset_index()

        # Merging SWC and labels
        i_merged = pd.merge(left=i_swc, 
                            right=flipped_labels.iloc[:,1:], 
                            left_on="node_id", 
                            right_on="node_id",
                            how="left")

        
        if os.path.exists(save_path) is False:
            os.mkdir(save_path)

        i_merged.to_csv(output_path)

    else:
        continue


In [ ]:
import pyarrow.dataset as ds
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
from joblib import Parallel, delayed

################
# > Define paths
path_simple1 = os.path.join("data", "input_swc", "simplified")
path_simple2 = os.path.join("data", "input_swc", "simplified_swc")
path_og = os.path.join("data", "input_swc", "sk_lod1_783_healed")
path_2process = path_simple1
parquet_path = os.path.join("data", "unified_clumpiness.parquet")
save_path = os.path.join("data", "output_results")

# Ensure the output directory exists before spawning workers
if not os.path.exists(save_path):
    os.makedirs(save_path)

############################
# > Get list of relevant neurons (from the parquet)
# Read the clumpiness file metadata once in the main process
dataset = ds.dataset(parquet_path, format="parquet")
table_neurons = dataset.to_table(columns=["neuron_id"]).to_pandas()
df_neuronsids = table_neurons['neuron_id'].unique() 

# List of SWC files
swc_simplified_dir = [i.split(".")[0] for i in os.listdir(path_simple1)]
swc_og_dir = [i.split(".")[0] for i in os.listdir(path_og)]

# Neurons to process
relv_files = np.intersect1d(df_neuronsids, swc_simplified_dir)

#################################################################################################################
# > Worker function for joblib that joins SWCs file with their appropriate clumpiness results by neuron_id label.
def join_swc_clumpiness(neuron_id: str , 
                        path_2process: str, 
                        save_path: str, 
                        parquet_path: str):

    """
    neuron_id: str -> neuron id as string file.
    path_2process: str -> path to the target swc files (to which the labels will be joined).
    save_path: str -> path to the save output folder.
    parquet_path -> path to the labels parquet file.
    
    """
    i = neuron_id
    output_path = os.path.join(save_path, f"{i}.csv")

    # Early exit if the file already exists
    if os.path.exists(output_path):
        return

    # Re-initialize the pyarrow dataset inside the worker
    # This prevents pickling/serialization errors across different CPU cores
    worker_dataset = ds.dataset(parquet_path, format="parquet")

    # Defining required path and loading swc file
    i_path = os.path.join(path_2process, f"{i}.csv")
    i_swc = pd.read_csv(i_path, index_col=0)
    
    # Filter dataset for the specific neuron_id
    i_label = worker_dataset.to_table(filter=(ds.field("neuron_id") == i)).to_pandas()
    
    # Optional safety check in case a neuron ID has no corresponding labels
    if i_label.empty:
        return
        
    i_label['node_id'] = i_label['node_id'].astype("int")

    # Creating a unified labels column named 'label'
    i_label.insert(loc=2, 
                   column="label",
                   value=i_label['property1'] + "_" + i_label['property2'])

    # Use pivot_table instead of pivot, and specify the index
    flipped_labels = i_label.pivot_table(index=['neuron_id', 'node_id'], 
                                         columns='label',
                                         values='value',
                                         aggfunc='first').reset_index()

    # Merging SWC and labels
    i_merged = pd.merge(left=i_swc, 
                        right=flipped_labels.iloc[:, 1:], 
                        left_on="node_id", 
                        right_on="node_id",
                        how="left")

    # Write out the file
    i_merged.to_csv(output_path)

######################################################################
# > Iterating over the SWC files in parallel
# n_jobs=-1 tells joblib to use all available CPU cores
_capture = Parallel(n_jobs=-1)(delayed(join_swc_clumpiness)(i, path_2process, save_path, parquet_path) 
                               for i in tqdm(relv_files, desc="Dispatching Tasks"))

In [ ]:
import os
from pathlib import Path


path_demo = os.path.join("root","dir_1","dir_2")
last_dir = last_folder = Path(path_demo).name
last_dir

In [ ]:
###############################
from scripts.preprocessing import simplify_swc_topology
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
from typing import Union
import networkx as nx
import pandas as pd
import polars as pl

def plot_neuron_topology(data: Union[str, pd.DataFrame], 
                         labels_df: pd.DataFrame = None, 
                         draw_node_id: bool = True):
    """
    Parses an SWC file or DataFrame and plots a customized hierarchical topological tree.
    
    Parameters:
    - data (str or pd.DataFrame): The string path to the .swc file OR a DataFrame representing the SWC.
    - labels_df (pd.DataFrame, optional): A DataFrame containing labels. 
      Assumes column 0 is the Node ID and column 1 is the label text. Defaults to None.
    - draw_node_id (bool, optional): Whether to display the numerical node ID next to the node. Defaults to True.
    """
    
    # ---------------------------------------------------------
    # 1. Parse SWC & Build Directed Graph
    # ---------------------------------------------------------
    G = nx.DiGraph()
    node_types = {} 

    if isinstance(data, pd.DataFrame):
        cols = [str(c).lower() for c in data.columns]
        
        id_col = data.columns[cols.index('id')] if 'id' in cols else data.columns[0]
        parent_col = data.columns[cols.index('parent')] if 'parent' in cols else data.columns[6]
        type_col = data.columns[cols.index('type')] if 'type' in cols else (data.columns[1] if len(data.columns) > 1 else None)

        for _, row in data.iterrows():
            node_id = int(row[id_col])
            parent_id = int(row[parent_col])
            
            G.add_node(node_id) 
            if parent_id != -1: 
                G.add_edge(parent_id, node_id)
                
            if type_col is not None:
                node_types[node_id] = row[type_col]

    elif isinstance(data, str):
        try:
            with open(data, "r") as f:
                for line in f:
                    if line.startswith("#") or not line.strip():
                        continue
                    parts = line.split()
                    if len(parts) >= 7:
                        node_id = int(parts[0])
                        node_type = parts[1]  
                        parent_id = int(parts[6])
                        
                        G.add_node(node_id) 
                        if parent_id != -1: 
                            G.add_edge(parent_id, node_id)
                            
                        node_types[node_id] = node_type
        except FileNotFoundError:
            print(f"Error: Could not find the file at {data}")
            return
    else:
        print("Error: Input data must be a file path string or a pandas DataFrame.")
        return
        
    if not G.nodes():
        print("Error: No valid nodes were found.")
        return

    # ---------------------------------------------------------
    # 2. Process Labels and Node Types
    # ---------------------------------------------------------
    label_dict = {}
    if labels_df is not None and not labels_df.empty:
        id_col = labels_df.columns[0]
        val_col = labels_df.columns[1]
        label_dict = dict(zip(labels_df[id_col], labels_df[val_col]))
        
        cols_lower = [str(c).lower() for c in labels_df.columns]
        if 'type' in cols_lower:
            type_col = labels_df.columns[cols_lower.index('type')]
            type_dict = dict(zip(labels_df[id_col], labels_df[type_col]))
            node_types.update(type_dict)

    # ---------------------------------------------------------
    # 3. Leaf-Weighted Hierarchical Layout Algorithm
    # ---------------------------------------------------------
    def get_hierarchy_pos(G, root, xcenter=0.5, vert_gap=1.0, vert_loc=0):
        def count_leaves(n):
            children = list(G.successors(n))
            if not children: return 1
            return sum(count_leaves(c) for c in children)
        
        pos = {}
        def _pos(node, xcenter, vert_loc, available_width):
            pos[node] = (xcenter, vert_loc)
            children = list(G.successors(node))
            if children:
                total_leaves = sum(count_leaves(c) for c in children)
                left_edge = xcenter - available_width / 2
                for child in children:
                    child_leaves = count_leaves(child)
                    child_width = available_width * (child_leaves / total_leaves)
                    child_xcenter = left_edge + child_width / 2
                    _pos(child, child_xcenter, vert_loc - vert_gap, child_width)
                    left_edge += child_width
        
        total_tree_leaves = count_leaves(root)
        _pos(root, xcenter, vert_loc, available_width=max(1.0, total_tree_leaves * 1.0))
        return pos

    roots = [n for n, d in G.in_degree() if d == 0]
    pos = {}
    x_offset = 0.0

    for root in roots:
        sub_pos = get_hierarchy_pos(G, root, xcenter=x_offset)
        pos.update(sub_pos)
        
        root_width = max([p[0] for p in sub_pos.values()]) - min([p[0] for p in sub_pos.values()])
        x_offset += root_width + 2.0 

    # ---------------------------------------------------------
    # 4. Geometry-Based Minimum Spacing Scaling
    # ---------------------------------------------------------
    total_leaves = sum(1 for n in G.nodes() if G.out_degree(n) == 0)
    max_depth = abs(min(y for x, y in pos.values())) if pos else 1
    
    min_space_per_level = 0.5   
    min_space_per_leaf = 0.8    
    
    fig_width = max(12, total_leaves * min_space_per_leaf)
    fig_height = max(12, max_depth * min_space_per_level)

    plt.figure(figsize=(fig_width, fig_height), facecolor='white')
    ax = plt.gca()
    ax.axis('off')

    # ---------------------------------------------------------
    # 5. Determine Node Colors & Final Text Labels
    # ---------------------------------------------------------
    # Identify unique label texts applied to leaf nodes
    unique_leaf_labels = set()
    leaf_label_map = {}
    
    for node in G.nodes():
        if G.out_degree(node) == 0:
            texts = []
            if node in label_dict:
                texts.append(str(label_dict[node]).strip())
            if node in node_types:
                leaf_val = node_types[node]
                if pd.notna(leaf_val) and str(leaf_val).strip().lower() not in ['nan', 'none', 'null', '']:
                    texts.append(str(leaf_val).strip())
            
            if texts:
                combined = " ".join(texts)
                unique_leaf_labels.add(combined)
                leaf_label_map[node] = combined

    # Create a mapping from unique leaf label -> specific color using standard colormap
    try:
        cmap = plt.colormaps.get_cmap('tab10')
    except AttributeError:
        cmap = plt.cm.get_cmap('tab10') # Fallback for older matplotlib versions
        
    color_palette = [cmap(i % 10) for i in range(len(unique_leaf_labels))]
    label_color_map = dict(zip(sorted(list(unique_leaf_labels)), color_palette))

    # Assign final text and node color iteratively
    final_labels = {}
    node_colors = []
    
    for node in G.nodes():
        id_text = str(node) if draw_node_id else ""
        node_color = 'white' # Default internal/unlabeled node color
        
        if G.out_degree(node) == 0 and node in leaf_label_map:
            label_text = leaf_label_map[node]
            node_color = label_color_map[label_text] # Apply the unique color mapping
        else:
            # Handle intermediate branches that happen to have custom dictionary labels
            label_text = str(label_dict[node]).strip() if node in label_dict else ""
            
        # Stack the ID on top of the label with a newline character
        if id_text and label_text:
            final_labels[node] = f"{id_text}\n{label_text}"
        elif id_text:
            final_labels[node] = id_text
        elif label_text:
            final_labels[node] = label_text
            
        node_colors.append(node_color)

    # ---------------------------------------------------------
    # 6. Render Graph
    # ---------------------------------------------------------
    # Edges first so they sit cleanly under the nodes
    nx.draw_networkx_edges(G, pos, width=2.5, edge_color='black', arrows=False)
    
    # Nodes filled with dynamically mapped colors
    nx.draw_networkx_nodes(G, pos, node_size=80, node_color=node_colors, edgecolors='black', linewidths=1.5)

    if final_labels:
        width_span = max(x for x, y in pos.values()) - min(x for x, y in pos.values()) if len(pos) > 1 else 1
        x_shift = width_span * 0.005 
        
        pos_labels = {k: (v[0] + x_shift, v[1]) for k, v in pos.items()}

        nx.draw_networkx_labels(
            G, 
            pos_labels, 
            labels=final_labels, 
            font_size=9, 
            font_color='darkred',
            font_weight='bold',
            horizontalalignment='left',
            verticalalignment='center'
        )

    # Automatically generate a legend if colored labels exist
    if label_color_map:
        handles = [mlines.Line2D([], [], color='white', marker='o', markerfacecolor=color, 
                                 markersize=10, markeredgecolor='black', label=lbl) 
                   for lbl, color in label_color_map.items()]
        # Position legend in the upper right outside of the immediate drawing area to prevent overlap
        plt.legend(handles=handles, title="Leaf Types", loc='upper right', fontsize=12, title_fontsize=14)

    plt.title("Annotated Neuron Topology", fontsize=18, pad=20)
    plt.tight_layout()
    plt.show()

In [3]:
# Paths
neuron_id = 720575940609102805
swc_path = os.path.join("data", "input_swc", "sk_lod1_783_healed", f"{neuron_id}.swc")
prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

# Load exactly the labels of the example swc file
parquet_labels = pl.scan_parquet(prquet_labels_path)

# Only 5 top rows of the parquet file (for inspection)
parquet_preview5 = parquet_labels.head(n=5).collect()

# Only the relevnt column in the parquet file
parquet_filt = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_id)).collect()

# Importing neuron and labels datasets
neuron_labels = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_id)).collect().to_pandas()
neuron_swc = pd.read_csv(swc_path, comment='#', header=None, sep=r'\s+', names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])

# Joining labels into neuron nodes
neuron_merged = pd.merge(left=neuron_swc, right=neuron_labels[["node_id", "type"]], left_on="node_id", right_on="node_id", how="outer")

swc_simple = simplify_swc_topology(neuron_merged,output_path=os.path.join("data", "test"), swc_name=720575940609102805)

plot_neuron_topology(neuron_swc)

OSError: Cannot save file into a non-existent directory: 'data\test'